# Visual Odometry

Task statement: given a video, estimate the camera's rotation and translation between consecutive frames.

1. Given:
    Each pixel has a form: $p=\begin{bmatrix} u \\ v \\ 1 \end{bmatrix}$

    Camera is calibrated, and intrinsic parameters matrix in known:

    $K=\begin{bmatrix}
        f_x & 0 & c_x \\
        0 & f_y & c_y \\
        0 & 0 & 1
    \end{bmatrix}$

    At least 5 pairs of features correspondence between frame 1 and frame 2:

    $p_{1,i} \leftrightarrow p_{2,i}, i=\overline{1,5}$

2. Transform each pixel into a normalized camera ray:

    $x=K^{-1}p$

3. Essential matrix definition. It encodes the relative rotation and translation between the two camera views.

    $P_2 = RP_1 + t$ – coordinate transformation from camera 1 to camera 2 of real-world point $P$.

    $E=[t]_{\times}R$ – Essential matrix definition

    $[t]_{\times}=\begin{bmatrix}
            0 & -t_z & t_y \\
            t_z & 0 & -t_x \\
            -t_y & t_x & 0
        \end{bmatrix}$ – the skew-symmetric matrix corresponding to the translation vector.

    $R$ - 3x3 rotation matrix

4. Essential matrix computation. Five points method.

   It should satisfy the constraint:

    $x_2^TEx_1=0$

    Equation is based on the fact that three vectors are coplanar if their scalar triple product is zero. Vectors $x_1, x_2, t$ should be coplanar (epipolar geometry triangle).

    Matrix $E$ has 5 degrees of freedom (3 for rotation and 2 for translation, because translation magnitude is unknown).
    We cannot solve five linear equations for $E$. Because we also need to enforce the singular values to have the form $(\sigma, \sigma, 0)$.
    The five-point method must enforce that structure, which makes the problem nonlinear (need to solve the polynom of 10th degree).

    ```python
    E, mask = cv2.findEssentialMat(
        pts1,
        pts2,
        K,
        method=cv2.RANSAC
    )
    ```

    Note: RANSAC is used to improve the robustness of the estimation, but we need more than 5 points for this method.

5. Decompose Essential matrix:

    1. First, apply SVD decomposition.

        $E=U\Sigma V^T$

        For an ideal Essential Matrix, the singular values have the special structure:

        $\Sigma=\begin{bmatrix}
            \sigma & 0 & 0 \\
            0 & \sigma & 0 \\
            0 & 0 & 0
        \end{bmatrix}$

       Because $E$ itself has an arbitrary scale, we normalize this to:

       $\Sigma=\begin{bmatrix}
            1 & 0 & 0 \\
            0 & 1 & 0 \\
            0 & 0 & 0
        \end{bmatrix}$
   2. Then, extract target components
         $t=\pm u_3$ – the third column of $U$, translation candidates.

         Construct a special matrix:

         $W=\begin{bmatrix}
                 0 & -1 & 0 \\
                 1 & 0 & 0 \\
                 0 & 0 & 1
             \end{bmatrix}$

         Extract $R$ candidates:

         $R_1=UWV^T$

         $R_2=UW^TV^T$

    ```python
    _, R, t, mask = cv2.recoverPose(
        E,
        pts1,
        pts2,
        K
    )
    ```

6. The cheirality check.

    Epipolar constraint by itself doesn't uniquely tell us which configuration is physically correct.
    Hence, after SVD decomposition we get: $candidate R_1, candidate R_2, candidate t, candidate -t$

    For each candidate $(R,t)$ we need to check whether it is physically correct:
    1. Triangulate some corresponding points.
    2. Check their depths.
    3. Choose the pose where the points are in front of both cameras.

    OpenCV's recoverPose() uses this approach to select the physically valid relative pose.

7. Point coordinate transformation. Translation vector properties.

    Compute $(R, t)$ for consecutive frame pairs.
    These relative transformations are composed to obtain the trajectory.

    Note: Each translation vector has unit length.

   There are two separate issues with translation vector:
    1. Absolute scale ambiguity: monocular VO cannot recover the real-world distance traveled from two images alone.
    2. Relative scale consistency: the translation estimated from different frame pairs might not be mutually consistent in scale.
    Multiframe algorithm could address this issue.

    **Note**: you must rotate the earlier translation into the newer coordinate frame before adding the next translation.
   Simply summing the translation vectors is generally incorrect.

   The valid approach to get coordinates of point $P$ in $n-th$ camera coordinate system:
   1. Compute pairwise transformations:

      $Frame \ 0 \rightarrow Frame \ 1: (R_{01}, t_{01})$

      $Frame \ 1 \rightarrow Frame \ 2: (R_{12}, t_{12})$
   2. Define relations:

      $P_1=R_{01}P_0+t_{01}$

      $P_2=R_{12}P_1+t_{12}$
   3. Substitute:

        $P_2=R_{12}(R_{01}P_0+t_{01})+t_{12} = R_{12}R_{01}P_0+R_{12}t_{01}+t_{12} \implies$

        $R_{02}=R_{12}R_{01}$

        $t_{02}=R_{12}t_{01}+t_{12}$

   Conclusion: you must rotate the earlier translation into the newer coordinate frame before adding the next translation.



1. Trajectory composition.

    Notions:
    - $k$ - index of a camera coordinate system

    - $w$ - world coordinate system

    Algorithm:

    1. Transformation of point coordinates from $i$ to $j$ coordinate system could be represented by a homogeneous transformation matrix:

        $T_{i,j} = \begin{bmatrix} R_{i, j} & t_{i, j} \\ 0 & 1 \end{bmatrix}$

        And performed as matrix multiplication:

        $\begin{bmatrix} P_1 \\ 1 \end{bmatrix} = T_{0,1}\begin{bmatrix} P_0 \\ 1 \end{bmatrix}$

   2. Choose camera 0 as the world coordinate system. Its pose relative to itself is the identity:

        $T_{0, w} = I_4 = \begin{bmatrix} I_3 & 0 \\ 0 & 1 \end{bmatrix}$

   3. Find $T_{w,k}$, which maps coordinates from camera $k$ into world coordinates $w$:

        $P_w = T_{0,w}P_0=T_{1,w}P_1=T_{1,w}T_{0,1}P_0$ – since $P$ is the same point, after mapping from different coordinate systems into $w$ it should have the same coordinates.

        Then:

        $T_{1,w} = T_{0,w} T^{-1}_{0,1}$

        And generally:

        $T_{k,w} = T_{k-1, w}T^{-1}_{k-1, k}$

        Inverse matrix:

        $T^{-1}_{k-1, k} = \begin{bmatrix} R_{k-1,k}^T & -R_{k-1,k}^Tt_{k-1,k} \\ 0 & 1 \end{bmatrix}$

        Therefore, the rotation and position updates are:

        $T_{k,w} = \begin{bmatrix} R_{k-1, w} & t_{k-1, w} \\ 0 & 1 \end{bmatrix} \begin{bmatrix} R_{k-1,k}^T & -R_{k-1,k}^Tt_{k-1,k} \\ 0 & 1 \end{bmatrix} = \begin{bmatrix} R_{k, w} & t_{k, w} \\ 0 & 1 \end{bmatrix}$

        $R_{k,w}=R_{k-1,w}R^T_{k-1,k}$

        $t_{k,w}= -R_{k,w}t_{k-1,k} + t_{k-1, w}$